# 1D ADI resonance vs CFL

Matches the Artemis PEC-soft / periodic comparison and the A–D electric-source scan, on a 1D Yee grid.

- $E$ at nodes $i$, $H$ at $i+\tfrac12$
- PEC: $E_0=E_N=0$, soft drive of the one-wavelength cavity mode
- Periodic: traveling-wave IC, no source

ADI E-source schemes (as in `algo.adi_e_excitation`):

| | placement |
|---|---|
| A | $S^{n+1/2}$ in the first half only |
| B | $S^{n+1/2}$ in the second half only |
| C | $\tfrac12 S^{n+1/2}$ in both halves |
| D | $\tfrac12 S^{n+1/4}$ then $\tfrac12 S^{n+3/4}$ |

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

C0 = 299792458.0
EPS0 = 8.8541878128e-12
MU0 = 1.25663706212e-6
Z0 = MU0 * C0

L = 8.0e-6
NZ = 2928
N_PERIODS = 16.0
E0 = 1.0
CFLS = [32.0, 64.0, 128.0, 256.0, 512.0]
SCHEMES = ("a", "b", "c", "d")
FFT_PAD = 8

dx = L / NZ
k = 2.0 * math.pi / L
kh = k * dx
f0 = C0 * k / (2.0 * math.pi)
omega_d = 2.0 * math.pi * f0
T_P = 1.0 / f0
t0_env = 3.0 * T_P

x_E_pec = np.linspace(0.0, L, NZ + 1)
x_E_per = np.arange(NZ) * dx
x_H = (np.arange(NZ) + 0.5) * dx
probe_pec = int(np.argmin(np.abs(x_E_pec - 0.25 * L)))
probe_per = int(np.argmin(np.abs(x_E_per - 0.25 * L)))

print(f"Nz={NZ}, k dz={kh:.4e}, f0={f0/1e12:.4f} THz")
print("CFL:", CFLS)

In [ ]:
def thomas(a, b, c, d):
    n = len(b)
    bc = np.array(b, float)
    dc = np.array(d, float)
    cc = np.array(c, float)
    for i in range(1, n):
        w = a[i - 1] / bc[i - 1]
        bc[i] -= w * cc[i - 1]
        dc[i] -= w * dc[i - 1]
    x = np.empty(n)
    x[-1] = dc[-1] / bc[-1]
    for i in range(n - 2, -1, -1):
        x[i] = (dc[i] - cc[i] * x[i + 1]) / bc[i]
    return x


def cyclic_thomas(a, b, c, d):
    n = len(d)
    gamma = -b[0]
    b1 = b.copy()
    b1[0] -= gamma
    b1[-1] -= a[0] * c[-1] / gamma
    u = np.zeros(n)
    u[0] = gamma
    u[-1] = c[-1]
    y = thomas(a[1:], b1, c[:-1], d)
    z = thomas(a[1:], b1, c[:-1], u)
    vn = a[0] / gamma
    fact = (y[0] + vn * y[-1]) / (1.0 + z[0] + vn * z[-1])
    return y - fact * z


def implicit_lhs_coeffs(n, gamma, periodic):
    off = gamma / dx**2
    a = np.full(n, -off)
    b = np.full(n, 1.0 + 2.0 * off)
    c = np.full(n, -off)
    if periodic:
        return a, b, c
    return a[1:], b, c[:-1]


def solve_E(rhs_interior, gamma, periodic):
    n = rhs_interior.size
    if periodic:
        a, b, c = implicit_lhs_coeffs(n, gamma, True)
        return cyclic_thomas(a, b, c, rhs_interior)
    a, b, c = implicit_lhs_coeffs(n, gamma, False)
    return thomas(a, b, c, rhs_interior)


def delta_H(H, periodic):
    if periodic:
        return (H - np.roll(H, 1)) / dx
    dH = np.zeros(H.size + 1)
    dH[1:-1] = (H[1:] - H[:-1]) / dx
    return dH


def delta_E(E, periodic):
    if periodic:
        return (np.roll(E, -1) - E) / dx
    return (E[1:] - E[:-1]) / dx


def adi_step(E, H, S1, S2, C_b, D_b, gamma, periodic):
    rhs = E + C_b * delta_H(H, periodic) + S1
    if periodic:
        E_half = solve_E(rhs, gamma, True)
    else:
        E_half = np.zeros_like(E)
        E_half[1:-1] = solve_E(rhs[1:-1], gamma, False)
    dE = delta_E(E_half, periodic)
    H_half = H + D_b * dE
    E_new = E_half + C_b * delta_H(H_half, periodic) + S2
    if not periodic:
        E_new[0] = 0.0
        E_new[-1] = 0.0
    H_new = H_half + D_b * dE
    return E_new, H_new

In [ ]:
def coeffs(cfl):
    dt = cfl * dx / C0
    nsteps = int(math.ceil(N_PERIODS / (f0 * dt)))
    C_b = dt / (2.0 * EPS0)
    D_b = dt / (2.0 * MU0)
    gamma = C_b * D_b
    return dt, nsteps, C_b, D_b, gamma


def S_E(x, t, dt):
    env = np.exp(-0.5 * ((t - t0_env) / T_P) ** 2)
    return E0 * (dt / T_P) * np.sin(k * x) * env * np.sin(omega_d * t)


def sources(scheme, t_n, dt, x):
    z = np.zeros_like(x)
    if scheme == "a":
        return S_E(x, t_n + 0.5 * dt, dt), z
    if scheme == "b":
        return z, S_E(x, t_n + 0.5 * dt, dt)
    if scheme == "c":
        s = 0.5 * S_E(x, t_n + 0.5 * dt, dt)
        return s, s
    if scheme == "d":
        return 0.5 * S_E(x, t_n + 0.25 * dt, dt), 0.5 * S_E(x, t_n + 0.75 * dt, dt)
    raise ValueError(scheme)


def run_pec(cfl, scheme):
    dt, nsteps, C_b, D_b, gamma = coeffs(cfl)
    E = np.zeros(NZ + 1)
    H = np.zeros(NZ)
    t = np.empty(nsteps + 1)
    e = np.empty(nsteps + 1)
    t[0] = 0.0
    e[0] = E[probe_pec]
    for n in range(nsteps):
        S1, S2 = sources(scheme, n * dt, dt, x_E_pec)
        E, H = adi_step(E, H, S1, S2, C_b, D_b, gamma, False)
        t[n + 1] = (n + 1) * dt
        e[n + 1] = E[probe_pec]
    return t, e


def run_periodic(cfl):
    dt, nsteps, C_b, D_b, gamma = coeffs(cfl)
    E = E0 * np.sin(k * x_E_per)
    H = -E0 / Z0 * np.sin(k * x_H)
    t = np.empty(nsteps + 1)
    e = np.empty(nsteps + 1)
    z = np.zeros_like(E)
    t[0] = 0.0
    e[0] = E[probe_per]
    for n in range(nsteps):
        E, H = adi_step(E, H, z, z, C_b, D_b, gamma, True)
        t[n + 1] = (n + 1) * dt
        e[n + 1] = E[probe_per]
    return t, e

In [ ]:
def last_half(times, signal):
    i0 = len(times) // 2
    return times[i0:], signal[i0:]


def compute_fft(times, signal, pad_factor=FFT_PAD):
    t, y = last_half(times, signal)
    y = y - np.mean(y)
    dt = float(np.median(np.diff(t)))
    n = len(y)
    n_fft = max(n, int(pad_factor) * n)
    spec = np.fft.rfft(y * np.hanning(n), n=n_fft)
    freqs = np.fft.rfftfreq(n_fft, d=dt)
    return freqs, np.abs(spec) / n


def peak_metrics(times, signal):
    freqs, amp = compute_fft(times, signal)
    i = 1 + int(np.argmax(amp[1:]))
    return float(amp[i]), float(freqs[i] / f0)


def analytical_f_over_f0(cfl):
    return 2.0 * math.atan(cfl * math.sin(0.5 * kh)) / (cfl * kh)


def detuning_amplitude(cfl, exp_prefactor=0.5):
    omega_adi = omega_d * analytical_f_over_f0(cfl)
    return math.exp(-exp_prefactor * T_P**2 * (omega_adi - omega_d) ** 2)


def carrier_sampling_factor(cfl):
    dt = cfl * dx / C0
    return math.cos(math.pi * f0 * dt / 2.0)


def scheme_c_source_factor(cfl):
    """Scheme C scale vs physical ∂tS: |F̂_C| / |i ω_ADI Ŝ| = |sinc(ω_ADI Δt/2)|."""
    dt = cfl * dx / C0
    omega_adi = omega_d * analytical_f_over_f0(cfl)
    return abs(math.sin(0.5 * omega_adi * dt)) / (0.5 * omega_adi * dt)


STYLES = {
    "a": ("C0", "x", r"A: $S^{n+1/2}$ first half"),
    "b": ("C1", "o", r"B: $S^{n+1/2}$ second half"),
    "c": ("C2", "s", r"C: $\frac{1}{2} S^{n+1/2}$ both"),
    "d": ("C3", "D", r"D: $\frac{1}{2} S^{n+1/4},\,\frac{1}{2} S^{n+3/4}$"),
}

In [ ]:
pec_series = {scheme: {} for scheme in SCHEMES}
pec_metrics = {scheme: {} for scheme in SCHEMES}
periodic_series = {}
periodic_ff0 = {}

print("=== PEC soft drive ===")
print(f"{'scheme':>6} {'CFL':>8} {'FFT peak':>12} {'f/f0':>10} {'f_ADI/f0':>10}")
for scheme in SCHEMES:
    for cfl in CFLS:
        t, e = run_pec(cfl, scheme)
        pec_series[scheme][cfl] = (t, e)
        peak, ff0 = peak_metrics(t, e)
        pec_metrics[scheme][cfl] = {"fft_peak": peak, "f_over_f0": ff0}
        print(f"{scheme:>6} {cfl:8g} {peak:12.4e} {ff0:10.6f} {analytical_f_over_f0(cfl):10.6f}")

print("\n=== Periodic traveling-wave IC ===")
print(f"{'CFL':>8} {'f/f0':>10} {'f_ADI/f0':>10}")
for cfl in CFLS:
    t, e = run_periodic(cfl)
    periodic_series[cfl] = (t, e)
    _, ff0 = peak_metrics(t, e)
    periodic_ff0[cfl] = ff0
    print(f"{cfl:8g} {ff0:10.6f} {analytical_f_over_f0(cfl):10.6f}")

In [ ]:
cmap = plt.cm.viridis
cfl_colors = {cfl: cmap(i / max(1, len(CFLS) - 1)) for i, cfl in enumerate(CFLS)}
kh_label = f"{kh:.3e}".replace("e-0", "e-")

fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.5), sharex="col")
panels = [
    ("pec", pec_series["c"], rf"PEC soft drive C ($N_z={NZ}$, $k\Delta x={kh_label}$)"),
    ("periodic", periodic_series, rf"Periodic sine IC ($N_z={NZ}$, $k\Delta x={kh_label}$)"),
]
for row, (mode, series, title) in enumerate(panels):
    ax_t, ax_f = axes[row]
    for cfl in CFLS:
        times, ey = series[cfl]
        ax_t.plot(times * f0, ey, color=cfl_colors[cfl], lw=1.2, label=fr"$S={cfl:g}$")
        freqs, amp = compute_fft(times, ey)
        ax_f.plot(freqs / f0, amp, color=cfl_colors[cfl], lw=1.4, label=fr"$S={cfl:g}$")
        ax_f.axvline(analytical_f_over_f0(cfl), color=cfl_colors[cfl], ls="--", lw=0.9, alpha=0.7)
    t_ref = series[CFLS[0]][0]
    ax_t.axvline(t_ref[len(t_ref) // 2] * f0, color="k", ls=":", lw=1.0, alpha=0.6)
    ax_t.set_ylabel(r"$E(x=L/4)$")
    ax_t.set_title(title + " — time")
    ax_t.grid(alpha=0.25)
    ax_t.legend(frameon=False, fontsize=8, ncol=2)
    ax_f.axvline(1.0, color="k", ls=":", lw=1.3, label=r"exact $f_0$")
    ax_f.set_xlim(0.0, 1.5)
    ax_f.set_ylabel(r"$|\mathrm{FFT}|/N$")
    ax_f.set_title(title + rf" — FFT (last half, pad$\times${FFT_PAD}, $/N$)")
    ax_f.grid(alpha=0.25)
    ax_f.legend(frameon=False, fontsize=8, ncol=2)

axes[1, 0].set_xlabel(r"$t\,f_0$")
axes[1, 1].set_xlabel(r"$f / f_0$")
fig.suptitle(rf"1D ADI: PEC soft C vs periodic, $k\Delta x={kh_label}$", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
s_max = max(CFLS)
s_adi = np.logspace(-2, np.log10(s_max * 1.05), 1600)
vp_adi = 2.0 * np.arctan(s_adi * np.sin(kh / 2.0)) / (s_adi * kh)

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(s_adi, vp_adi, color="k", lw=2, label=fr"ADI ana. $k\Delta x={kh_label}$")
ax.plot(
    CFLS,
    [pec_metrics["c"][c]["f_over_f0"] for c in CFLS],
    "x",
    color="C0",
    ms=8,
    mew=1.6,
    label="PEC soft C",
)
ax.plot(
    CFLS,
    [periodic_ff0[c] for c in CFLS],
    "o",
    color="C1",
    ms=8,
    mew=1.6,
    label="Periodic IC",
)
ax.axhline(1.0, color="0.4", ls="--", lw=1.5, label="Exact")
ax.set_xscale("log")
ax.set_xlim(min(CFLS) * 0.7, s_max * 1.3)
ax.set_ylim(0.75, 1.03)
ax.set_xlabel(r"CFL number $S=c\Delta t/\Delta x$")
ax.set_ylabel(r"Normalized frequency $f/f_0$ ($=v_p/c$)")
ax.set_title("ADI numerical dispersion: 1D Yee vs analytical")
ax.grid(alpha=0.25, which="both")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
x = np.asarray(CFLS)

for scheme, (color, marker, label) in STYLES.items():
    r = pec_metrics[scheme]
    axes[0].loglog(x, [r[c]["fft_peak"] for c in CFLS], marker + "-", color=color, ms=7, label=label)
    axes[1].semilogx(x, [r[c]["f_over_f0"] for c in CFLS], marker + "-", color=color, ms=7, label=label)

ref = pec_metrics["c"][CFLS[0]]["fft_peak"]
s_amp = np.logspace(np.log10(min(CFLS)), np.log10(max(CFLS)), 200)
a_half = np.array([detuning_amplitude(c, 0.5) for c in s_amp])
a_full = np.array([detuning_amplitude(c, 1.0) for c in s_amp])
a_corr = np.array([detuning_amplitude(c, 0.5) * carrier_sampling_factor(c) for c in s_amp])
a_c = np.array([detuning_amplitude(c, 0.5) * scheme_c_source_factor(c) for c in s_amp])
axes[0].loglog(
    s_amp, a_half * (ref / detuning_amplitude(CFLS[0], 0.5)),
    "k--", lw=1.5, label=r"$\exp[-T_P^2(\omega_{\mathrm{ADI}}-\omega_d)^2/2]$",
)
axes[0].loglog(
    s_amp, a_full * (ref / detuning_amplitude(CFLS[0], 1.0)),
    "k:", lw=1.5, label=r"$\exp[-T_P^2(\omega_{\mathrm{ADI}}-\omega_d)^2]$",
)
axes[0].loglog(
    s_amp, a_corr * (ref / (detuning_amplitude(CFLS[0], 0.5) * carrier_sampling_factor(CFLS[0]))),
    "k-.", lw=1.5,
    label=r"$\exp[-T_P^2(\omega_{\mathrm{ADI}}-\omega_d)^2/2]\,\cos(\omega_d\Delta t/4)$",
)
axes[0].loglog(
    s_amp,
    a_c * (ref / (detuning_amplitude(CFLS[0], 0.5) * scheme_c_source_factor(CFLS[0]))),
    color="C2",
    ls="--",
    lw=1.5,
    label=r"$\exp[-T_P^2(\omega_{\mathrm{ADI}}-\omega_d)^2/2]\,\left|\frac{\sin(\omega_{\mathrm{ADI}}\Delta t/2)}{\omega_{\mathrm{ADI}}\Delta t/2}\right|$",
)

s = np.logspace(np.log10(min(CFLS)), np.log10(max(CFLS)), 200)
axes[1].plot(s, 2 * np.arctan(s * np.sin(kh / 2)) / (s * kh), "k--", lw=1.5, label="ADI analytic")
axes[1].axhline(1.0, color="0.5", ls=":", lw=1)

axes[0].set_xlabel("CFL $S$")
axes[0].set_ylabel(r"$|\mathrm{FFT}|/N$ peak")
axes[0].set_title("Spectral peak magnitude")
axes[0].grid(alpha=0.25, which="both")
axes[0].legend(frameon=False, fontsize=7)

axes[1].set_xlabel("CFL $S$")
axes[1].set_ylabel(r"$f_\mathrm{peak}/f_0$")
axes[1].set_title("Frequency (dispersion)")
axes[1].grid(alpha=0.25, which="both")
axes[1].legend(frameon=False, fontsize=8)

fig.suptitle(rf"PEC soft drive, ADI E source A–D ($k\Delta x={kh_label}$)", fontsize=12)
fig.tight_layout()
plt.show()

print("\n=== CFL sensitivity (max/min FFT peak) ===")
for scheme in SCHEMES:
    peaks = [pec_metrics[scheme][c]["fft_peak"] for c in CFLS]
    freqs = [pec_metrics[scheme][c]["f_over_f0"] for c in CFLS]
    print(
        f"soft {scheme}  FFT ratio={max(peaks)/min(peaks):.3g}  "
        f"f/f0 spread={max(freqs)-min(freqs):.4f}"
    )

In [ ]:
# Modal q-boost for soft drive S ∝ sin(k x): compare measured peaks to draft |F̂| with q ≠ 0.

kh_label = f"{kh:.3e}".replace("e-0", "e-")
lam_E2 = -(4.0 / dx**2) * math.sin(0.5 * kh) ** 2  # δ_x^{E,2} sin(kx) = λ sin(kx)


def gamma_of(cfl):
    dt = cfl * dx / C0
    return (C0 * dt / 2.0) ** 2  # = C_b D_b


def q_of(cfl):
    return -gamma_of(cfl) * lam_E2


def theta_adi(cfl):
    return 2.0 * math.atan(cfl * math.sin(0.5 * kh))  # ω_ADI Δt


def Fmag_over_S_dt(scheme, cfl, q=None):
    """|F̂| / (|Ŝ|/Δt) from draft E-source FT, at θ = ω_ADI Δt."""
    th = theta_adi(cfl)
    qq = q_of(cfl) if q is None else q
    s2, c2 = math.sin(th / 2.0), math.cos(th / 2.0)
    if scheme in ("a", "b"):
        # |F̂_mid,1| = |2Ŝ/Δt| √[sin²(θ/2) + q² cos²(θ/2)]
        return 2.0 * math.sqrt(s2**2 + qq**2 * c2**2)
    if scheme == "c":
        # |F̂_mid,2| = |2Ŝ/Δt sin(θ/2)|
        return 2.0 * abs(s2)
    if scheme == "d":
        # |F̂_quarter| = |Ŝ/Δt| |(1-q)sin(θ/4)+(1+q)sin(3θ/4)|
        return abs((1.0 - qq) * math.sin(th / 4.0) + (1.0 + qq) * math.sin(3.0 * th / 4.0))
    raise ValueError(scheme)


print("=== modal q (excitation sin(kx)) ===")
print(f"{'CFL':>8} {'q':>10} {'θ_ADI':>10} {'|FA|/|FC|':>12} {'|FD|/|FC|':>12} {'peakA/C':>10} {'peakB/C':>10} {'peakD/C':>10}")
for cfl in CFLS:
    fa, fc, fd = (Fmag_over_S_dt(s, cfl) for s in ("a", "c", "d"))
    pc = pec_metrics["c"][cfl]["fft_peak"]
    print(
        f"{cfl:8g} {q_of(cfl):10.4f} {theta_adi(cfl):10.4f} "
        f"{fa/fc:12.6f} {fd/fc:12.6f} "
        f"{pec_metrics['a'][cfl]['fft_peak']/pc:10.6f} "
        f"{pec_metrics['b'][cfl]['fft_peak']/pc:10.6f} "
        f"{pec_metrics['d'][cfl]['fft_peak']/pc:10.6f}"
    )

# Absolute |F̂| = (|Ŝ|/Δt) × (dimensionless factor). Ŝ is Δt-independent
# (notebook soft source already includes one Δt), so keep the 1/Δt.
def Fmag(scheme, cfl, q=None):
    dt = cfl * dx / C0
    return Fmag_over_S_dt(scheme, cfl, q=q) / dt


# Reference curves: detuning × |F̂|, anchored to measured C at lowest CFL.
cfl_ref = CFLS[0]
ref_peak_c = pec_metrics["c"][cfl_ref]["fft_peak"]
norm0 = detuning_amplitude(cfl_ref, 0.5) * Fmag("c", cfl_ref)


def amp_theory(scheme, cfl, q=None):
    return (
        ref_peak_c
        * detuning_amplitude(cfl, 0.5)
        * Fmag(scheme, cfl, q=q)
        / norm0
    )


s_amp = np.logspace(np.log10(min(CFLS)), np.log10(max(CFLS)), 300)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

# Absolute FFT peaks + q-aware (and q=0) references
for scheme, (color, marker, label) in STYLES.items():
    axes[0].loglog(
        CFLS,
        [pec_metrics[scheme][c]["fft_peak"] for c in CFLS],
        marker,
        color=color,
        ms=8,
        label=label,
    )

for scheme, color, ls, lab in [
    (
        "c",
        "C2",
        "-",
        r"$e^{-(T_P^2/2)(\omega_{\mathrm{ADI}}(S)-\omega_d)^2}"
        r"\left|\frac{2i\widehat{S}}{\Delta t}\sin(\theta/2)\right|$",
    ),
    (
        "a",
        "C0",
        "--",
        r"$e^{-(T_P^2/2)(\omega_{\mathrm{ADI}}(S)-\omega_d)^2}"
        r"\left|\frac{2\widehat{S}}{\Delta t}"
        r"\left[i\sin(\theta/2)-q\cos(\theta/2)\right]\right|$",
    ),
    (
        "d",
        "C3",
        "-.",
        r"$e^{-(T_P^2/2)(\omega_{\mathrm{ADI}}(S)-\omega_d)^2}"
        r"\left|\frac{i\widehat{S}}{\Delta t}\left[(1-q)\sin(\theta/4)"
        r"+(1+q)\sin(3\theta/4)\right]\right|$",
    ),
]:
    axes[0].loglog(s_amp, [amp_theory(scheme, c) for c in s_amp], color=color, ls=ls, lw=1.6, label=lab)

axes[0].set_xlabel("CFL $S$")
axes[0].set_ylabel(r"$|\mathrm{FFT}|/N$ peak")
axes[0].set_title(
    r"Peaks $\propto$ detuning $\times\left|\widehat{F}\right|$"
    r" ($\theta=\omega_{\mathrm{ADI}}\Delta t$, $q$ modal)"
)
axes[0].grid(alpha=0.25, which="both")
axes[0].legend(frameon=False, fontsize=6.5)

# Relative to C: isolates q-boost from overall detuning
axes[1].semilogx(CFLS, np.ones(len(CFLS)), "s", color="C2", ms=8, label="measured C/C")
for scheme, color, marker in [("a", "C0", "x"), ("b", "C1", "o"), ("d", "C3", "D")]:
    axes[1].semilogx(
        CFLS,
        [pec_metrics[scheme][c]["fft_peak"] / pec_metrics["c"][c]["fft_peak"] for c in CFLS],
        marker,
        color=color,
        ms=8,
        label=f"measured {scheme.upper()}/C",
    )

axes[1].semilogx(
    s_amp,
    [Fmag_over_S_dt("a", c) / Fmag_over_S_dt("c", c) for c in s_amp],
    "C0--",
    lw=1.6,
    label=r"$\left|\widehat{F}_{\mathrm{mid},1}\right|/\left|\widehat{F}_{\mathrm{mid},2}\right|$"
    r" $=\sqrt{1+q^2\cot^2(\theta/2)}$",
)
axes[1].semilogx(
    s_amp,
    [Fmag_over_S_dt("d", c) / Fmag_over_S_dt("c", c) for c in s_amp],
    "C3-.",
    lw=1.6,
    label=r"$\left|\widehat{F}_{\mathrm{quarter}}\right|/\left|\widehat{F}_{\mathrm{mid},2}\right|$"
    r" $=\left|\cos(\theta/4)+q\cos(\theta/2)/(2\cos(\theta/4))\right|$",
)

axes[1].set_xlabel("CFL $S$")
axes[1].set_ylabel("peak / peak$_C$")
axes[1].set_title(r"Relative boost (shared detuning cancels)")
axes[1].grid(alpha=0.25, which="both")
axes[1].legend(frameon=False, fontsize=6.5)

fig.suptitle(
    rf"Modal $q$-boost: $\lambda={lam_E2:.4e}$, $q=-\gamma\lambda$, $\theta=\omega_{{\mathrm{{ADI}}}}\Delta t$ ($k\Delta x={kh_label}$)",
    fontsize=11,
)
fig.tight_layout()
plt.savefig("modal_q_boost.png", dpi=300)
